# Week 2 示例：Sklearn + PyTorch 基础

- 作者：共享仓库示例
- 日期：2026-07-14
- 来源：`暑期居家集训学习计划.md` → Week 2 → Sklearn 与 PyTorch 基础
- 适用周次：Week 2
- 分类：Sklearn / PyTorch
- 关键词：Pipeline、GridSearchCV、分类评估、autograd
- 运行环境：Python 3.10+、NumPy、scikit-learn、PyTorch
- 适合读者：已经会基本 Python 和 NumPy 的初学者

## 学习目标

1. 用 Pipeline 组织预处理、训练和评估。
2. 用 GridSearchCV 搜索超参数。
3. 用 PyTorch autograd 观察反向传播得到的梯度。

> 这是从《实验室新生暑期居家集训学习计划》抽取的最小示例，不包含完整的 20 Newsgroups 实验和 MNIST 训练。

## 本 Notebook 演示

1. 用 Pipeline 完成数据预处理、分类和评估。
2. 用 GridSearchCV 搜索一个超参数。
3. 用 PyTorch autograd 观察反向传播得到的梯度。


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import torch
except ImportError:
    torch = None

print('NumPy:', np.__version__)
if torch is None:
    print('PyTorch 未安装，跳过 PyTorch 单元。')
else:
    print('PyTorch:', torch.__version__)

## 1. Pipeline：预处理、训练和评估

《实验室新生暑期居家集训学习计划》使用 `TfidfVectorizer` 将 20 Newsgroups 文本转换为特征。这里使用可离线生成的分类数据，保留相同的 `train_test_split → Pipeline → 指标评估` 流程。

In [ ]:
X, y = make_classification(
    n_samples=600, n_features=12, n_informative=6,
    n_redundant=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))

## 2. GridSearchCV：只在训练集上选择超参数

真实实验中可以把这里的分类器替换成《实验室新生暑期居家集训学习计划》中的 SVM、Random Forest 或 Gradient Boosting，并保留交叉验证流程。

In [ ]:
grid = GridSearchCV(
    pipe,
    param_grid={'clf__C': [0.01, 0.1, 1, 10]},
    cv=5, scoring='f1', n_jobs=-1
)
grid.fit(X_train, y_train)
print('Best parameters:', grid.best_params_)
print('Best CV F1:', round(grid.best_score_, 4))

## 3. PyTorch autograd：计算图与梯度

这是原文中 `requires_grad=True`、`backward()` 和梯度清零示例的最小版本。

In [ ]:
if torch is None:
    print('请先安装 PyTorch，再运行这一节。')
else:
    x = torch.tensor(2.0, requires_grad=True)
    y = torch.tensor(3.0, requires_grad=True)
    z = x**2 + 2 * x * y + y**3
    z.backward()

    print('z =', z.item())
    print('dz/dx =', x.grad.item())
    print('dz/dy =', y.grad.item())

    # 训练中每次反向传播前都需要清空旧梯度。
    x.grad.zero_()
    y.grad.zero_()

## 小结

完整周作业还应补充 20 Newsgroups 文本分类、ROC 曲线、混淆矩阵、MNIST 训练循环和 LaTeX 报告。